# Sensitivity grid — three anchors instead of six

The same six calibrations, `birt` -> `bmi` -> `flat` in each, on the `_small` design files.
**Generated by `notebooks/make_design_notebooks.py` — edit the generator,
not this file.** All logic is in `mice_imp/run_grid.py`.

## What this varies — read this before quoting any number from it

The `_small` files are **not a smaller sample.** Same 1363 respondents, same
128 columns, same form sizes 455 / 454 / 454. What changes is the anchor set:
**three anchors (`DA054`, `DA058`, `DA066`) where the main delivery carries
six.** Verified against the delivered files on 2026-08-28.

| design | overlap 1&2 / 1&3 / 2&3 — main | reduced anchor |
|---|---|---|
| CommonItem | 6 / 6 / 6 | **3 / 3 / 3** |
| Binterlaced | 2 / 2 / 2 | **1 / 1 / 1** |
| Standard | 3 / 3 / 0 | **1 / 2 / 0** |

So this is a sensitivity analysis on **linking strength**, not on sample size,
and it should be described that way in the write-up — "small" is the
collaborators' filename, not what is being varied. It is also a direct probe of
**open issue B**: two of the three surviving anchors are the correlated pair
`DA054` / `DA058` (r = 0.840), and `DA066` correlates 0.88 with `DA053` and
0.876 with `DA060`. Three anchors that are not independent carry less joint
information than three anchors would suggest, and Binterlaced drops to a
**single** shared item per form pair.

All three anchors are `DA*` items, so `nobike` leaves the anchor set untouched
in this variant exactly as it does in the main one, and the two item sets stay
comparable.

### Preflight check 2b FAILs here — decide before trusting the output

Run on 2026-08-28, all three files fail category sparsity: **10–11 items have a
category with fewer than 20 responses** (`DA061`, `DA068`, `DA053`, `Bike_1`,
`DA069_r`, `Bike_6`, `Bike_7`, `DA067`, `DA073`, `DA060`, and `DA066` in
Standard and Binterlaced), against 8–9 items in the main files. Thinner
anchoring spreads the same responses across fewer shared items, so open issue E
is *worse* here, not better. Expect divergent transitions and R-hat failures
that look like a sampler problem and are not one. This is the standing issue-E
decision (collapse the category, tighten the threshold prior, or document those
parameters as not interpretable) — it is not resolved by running this grid.

Order is CommonItem, Binterlaced, Standard, and `full` before `nobike`:
increasing difficulty, so a failure lands where there is the most signal to
diagnose it. `standard/nobike` is the thinnest cell in the grid and runs last.

The runner is **resumable** — a run whose three arm `.npz` files already
exist is skipped, so an interrupted weekend continues rather than restarting
— **isolated**, in that one run raising does not stop the other five, and
**logged**, with each run's output captured to
`results_reduced_anchor/<design>/<itemset>/run.log` and `results_reduced_anchor/grid_summary.csv`
rewritten after every run.

Everything this notebook writes goes under **`results_reduced_anchor/`**, and every
exported CSV row carries `variant = "reduced_anchor"`, so these estimates cannot be
pooled with the other grid's by accident.

## For a weekend run, prefer the terminal

A notebook kernel is the wrong container for something that has to survive
until Monday: a closed laptop, a suspended session or a dropped connection
takes the kernel and the run with it. From a terminal instead:

```
nohup python -m mice_imp.run_grid --variant reduced_anchor > results_reduced_anchor/grid.out 2>&1 &
tail -f results_reduced_anchor/grid.out
```

Same script, same resumability. Use the cells below to watch progress, or to
run a single cell of the grid interactively.

## Smoke first

Run the wiring check before committing a weekend to a full pass — it is
2 chains of 50/50 with a
20-epoch ADVI warm-start, against 4 x
3000/2000 for the real thing. It writes to the same
directories, so a real run afterwards needs `--force` (or delete the smoke
`.npz` files).

It is sized to answer *does the plumbing work*, not *what is the answer*: at
50 warmup draws the R-hats and the noise floor mean nothing.
Read only the three checks below off it. The floor on how fast it can go is
the B-MI imputation fit, which does not shrink with chain length; a re-run in
a directory that already has `imputation_model/` reuses it.

Override any of it with `--num-chains`, `--num-warmup`, `--num-samples`,
`--advi-iterations`.

Three things to read off it, each of which silently invalidates a run:

- `Initializing from ADVI solution` must appear for **all three** arms. If it
  appears for some and not others, the `flat`-vs-`birt` gap is measuring
  initialisation rather than Monte Carlo noise, and the noise floor is
  meaningless (CLAUDE.md fact 17).
- the **B-MI flat-fallback rate** must be low. Every fallback downgrades that
  cell to B-IRT, so a high rate means B-MI is not really running (fact 3).
- `flat` and `birt` must land on top of each other. They target the same
  posterior by construction; a systematic gap means the plumbing is wrong
  (fact 2).

In [ ]:
import subprocess, sys
from pathlib import Path

REPO = Path("/home/luis/CGE/mice_imp")

def run_grid(*flags):
    """Run mice_imp.run_grid as a subprocess, streaming its output."""
    cmd = [sys.executable, "-m", "mice_imp.run_grid", *flags]
    print(" ".join(cmd), flush=True)
    proc = subprocess.Popen(cmd, cwd=REPO, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end="")
    return proc.wait()


# Wiring check only -- uncomment to run it. It writes .npz files, so the
# full pass below would then need --force.
# run_grid("--smoke", "--variant", "reduced_anchor")

## The full grid

4 chains, 3000 warmup, 2000 samples, six times over. Only start this once
the smoke output above looks right.

No `--force` here: `results_reduced_anchor/` starts empty, so nothing is
skipped. Add it only after a smoke run has put `.npz` files there.

In [ ]:
run_grid("--variant", "reduced_anchor")

## Progress

Safe to run while the grid is still going: `results_reduced_anchor/grid_summary.csv` is
rewritten after every run, and each run's `run.log` is flushed as it goes.

In [ ]:
import pandas as pd

summary = pd.read_csv(REPO / "results_reduced_anchor" / "grid_summary.csv")
cols = [c for c in ["design", "variant", "itemset", "n_items", "status", "minutes",
                    "birt_max_rhat", "bmi_max_rhat", "flat_max_rhat",
                    "bmi_fallback_pct", "advi_init_count",
                    "theta_rmse_flat_vs_birt", "theta_rmse_bmi_vs_birt"]
        if c in summary.columns]
summary[cols]

## Re-run one cell

`--only` takes `design/itemset`, e.g. after fixing whatever made a single
run fail. Keep `--variant reduced_anchor` on it — without that the re-run reads the
other variant's data and writes over the other variant's results:

```
run_grid("--only", "standard/nobike", "--variant", "reduced_anchor", "--force")
```

In [ ]:
# run_grid("--only", "standard/nobike", "--variant", "reduced_anchor", "--force")